# Skills para modelos de lenguaje: laboratorio en Google Colab

Este notebook crea una skill reutilizable para análisis estratégico de ventas, genera datos de ejemplo, calcula KPI, produce gráficas y prepara archivos para cargarlos manualmente en ChatGPT Free.

**Limitación:** la skill se carga como contexto de una conversación; no se instala permanentemente en ChatGPT Free.




## ¿Por qué son importantes las skills?

Una skill es un conjunto de instrucciones que le dice al modelo cómo debe trabajar en una tarea específica.

Un modelo de lenguaje puede responder sobre muchos temas, pero una respuesta general no siempre es suficiente. Por ejemplo, para analizar ventas necesitamos que el modelo revise los datos, calcule indicadores, explique sus resultados y no invente información. La skill le da ese método de trabajo.

Las skills son relevantes porque ayudan a:

- obtener respuestas más ordenadas;
- repetir el mismo proceso en distintos archivos;
- reducir errores y respuestas inventadas;
- adaptar el modelo a una profesión o área de trabajo;
- separar una tarea grande en pasos claros;
- hacer que el resultado sea más fácil de revisar.

Una skill no cambia el entrenamiento del modelo. Funciona como una guía que se carga en el contexto de la conversación. En este ejemplo, la skill se guarda en un archivo y después se sube manualmente a ChatGPT Free.




## 1. Preparación




## Explicación del bloque 1: preparar el entorno

Este bloque importa las herramientas que usaremos.

- **NumPy** ayuda a crear números aleatorios para el ejemplo.
- **pandas** organiza los datos en tablas.
- **Matplotlib** y **seaborn** crean gráficas.
- **display** muestra tablas y texto de forma más clara.
- La semilla 42 hace que los datos de ejemplo sean iguales cada vez que se ejecuta el notebook.

No se usa una API ni una contraseña. Por eso el ejemplo puede correr en Colab sin pagar por una conexión a un modelo.




In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
np.random.seed(42)
sns.set_theme(style="whitegrid")
print("Entorno listo: no requiere API ni credenciales.")



## 2. Datos sintéticos reproducibles




## Explicación del bloque 2: crear datos de ejemplo

Aquí se fabrica una pequeña tabla de ventas. Se crean meses, productos, regiones y vendedores.

Después, los ciclos **for** combinan cada mes, región y producto. Para cada combinación se calculan unidades, ventas, margen y objetivo. Los números son simulados, pero tienen una estructura parecida a la de un archivo real.

La tabla se guarda como **ventas_demo.csv**. Este archivo servirá después para probar la skill en ChatGPT Free.




In [ ]:
meses = pd.date_range("2025-01-01", periods=12, freq="MS")
productos = ["Analítica", "Automatización", "Capacitación", "Consultoría"]
regiones = ["Norte", "Centro", "Occidente", "Sureste"]
vendedores = ["Ana", "Bruno", "Carla", "Diego", "Elena"]
precios = {"Analítica":1800, "Automatización":2400, "Capacitación":950, "Consultoría":3200}
filas = []
for fecha in meses:
    for region in regiones:
        for producto in productos:
            unidades = np.random.randint(8, 45)
            ventas = unidades * precios[producto] * np.random.uniform(.88, 1.12)
            filas.append({"fecha":fecha, "region":region, "producto":producto,
                          "vendedor":np.random.choice(vendedores), "unidades":unidades,
                          "ventas":round(ventas,2), "margen":round(ventas*np.random.uniform(.22,.46),2),
                          "objetivo":round(ventas*np.random.uniform(.90,1.10),2)})
ventas = pd.DataFrame(filas)
ventas.to_csv("ventas_demo.csv", index=False)
display(ventas.head())
print(f"Registros: {len(ventas):,}")



## 3. Validación, KPI y visualización




## Explicación del bloque 3: revisar, calcular y mostrar

Este bloque hace tres trabajos.

Primero revisa que estén las columnas necesarias y cuenta datos vacíos y duplicados. Esto es importante porque una conclusión puede ser incorrecta si los datos tienen problemas.

Después calcula cinco indicadores: ventas totales, margen total, unidades, venta promedio y cumplimiento del objetivo.

Finalmente agrupa las ventas por mes y por producto. Con esos grupos crea dos gráficas: una compara ventas contra objetivo y la otra compara productos. Las gráficas ayudan a ver tendencias sin leer toda la tabla.




In [ ]:
requeridas = {"fecha","region","producto","vendedor","unidades","ventas","margen","objetivo"}
assert not (requeridas - set(ventas.columns)), "Faltan columnas."
print("Nulos:", int(ventas.isna().sum().sum()))
print("Duplicados:", int(ventas.duplicated().sum()))
ventas_totales = ventas.ventas.sum()
margen_total = ventas.margen.sum()
cumplimiento = ventas_totales / ventas.objetivo.sum()
kpis = pd.DataFrame({"Indicador":["Ventas totales","Margen total","Unidades","Ticket promedio","Cumplimiento"],
 "Valor":[ventas_totales,margen_total,ventas.unidades.sum(),ventas_totales/len(ventas),cumplimiento]})
kpis["Valor mostrado"] = kpis.Valor.map(lambda x: f"{x:,.2f}")
kpis.loc[kpis.Indicador=="Cumplimiento", "Valor mostrado"] = f"{cumplimiento:.1%}"
display(kpis[["Indicador","Valor mostrado"]])
mensual = ventas.groupby("fecha",as_index=False).agg(ventas=("ventas","sum"),objetivo=("objetivo","sum"))
producto = ventas.groupby("producto",as_index=False).agg(ventas=("ventas","sum"))
fig, ax = plt.subplots(1,2,figsize=(15,5))
ax[0].plot(mensual.fecha,mensual.ventas,marker="o",label="Ventas")
ax[0].plot(mensual.fecha,mensual.objetivo,linestyle="--",label="Objetivo")
ax[0].set(title="Ventas mensuales frente al objetivo",xlabel="Mes",ylabel="Monto"); ax[0].tick_params(axis="x",rotation=45); ax[0].legend()
sns.barplot(data=producto.sort_values("ventas",ascending=False),x="ventas",y="producto",ax=ax[1])
ax[1].set(title="Ventas por producto",xlabel="Ventas",ylabel="")
plt.tight_layout(); plt.show()



## 4. Crear la skill elaborada




## Explicación del bloque 4: crear la skill

La variable **skill** contiene las instrucciones que usará ChatGPT. Al principio del texto se agrega un **frontmatter**, que es un pequeño encabezado YAML entre dos líneas con tres guiones.

El frontmatter identifica la skill:

- **name**: nombre corto de la skill.
- **description**: explica para qué sirve y cuándo puede usarse.

Después del frontmatter aparecen las instrucciones normales: el papel del modelo, los pasos, las reglas y el formato de respuesta.

Al final, el código revisa que existan las cuatro partes básicas del encabezado. Si falta alguna, Colab muestra un error. Si todo está bien, guarda el archivo **SKILL.md**.




In [ ]:
skill = '''---
name: skill-analista-ventas
description: Skill-analista-ventas para analizar CSV, Excel y tablas de ventas. Actívala cuando el usuario pida análisis de ventas, KPI comerciales, margen, clientes, productos, regiones, vendedores o recomendaciones.
---

# Analista Estratégico de Ventas

## Identificación

El nombre exacto de esta skill es **skill-analista-ventas**. Los nombres alternativos son **analista de ventas**, **análisis de ventas** y **skill_analista_ventas**.

## Cuándo usar esta skill

Activa esta skill cuando el usuario diga: "usa la skill de ventas", "usa skill-analista-ventas", "analiza este CSV de ventas", "calcula los KPI de ventas" o pida recomendaciones comerciales.

Cuando la actives, comienza la respuesta indicando: **Skill utilizada: skill-analista-ventas**.

## Rol
Actúa como analista senior de ventas, rentabilidad y estrategia comercial.

## Activación
Usa esta skill cuando el usuario proporcione CSV, Excel o tablas de ventas, clientes, productos, regiones, vendedores, márgenes u objetivos.

## Proceso obligatorio
1. Inspecciona columnas, tipos, periodo, nulos, duplicados y valores negativos.
2. Explica problemas de calidad antes de concluir.
3. Calcula ventas totales, margen, unidades, ticket promedio y cumplimiento.
4. Compara por periodo, producto, región y vendedor cuando existan.
5. Separa hechos, cálculos e inferencias.
6. Propón recomendaciones priorizadas y medibles.

## Reglas
- No inventes datos faltantes.
- Declara fórmulas y supuestos.
- No presentes estimaciones como hechos.
- Advierte cuando una conclusión dependa de pocos registros.
- Usa moneda, unidades y periodo explícitos.

## Formato de salida
### Resumen ejecutivo
Máximo cinco conclusiones respaldadas por datos.

### Calidad de datos
Problema, impacto y acción correctiva.

### KPI principales
Indicador, valor, comparación, fórmula e interpretación.

### Hallazgos
Por producto, región, periodo y vendedor.

### Recomendaciones priorizadas
Acción, problema, responsable, prioridad, impacto y métrica.

### Incertidumbres
Preguntas pendientes y nivel de confianza: Alto, Medio o Bajo.
'''

open("SKILL.md", "w", encoding="utf-8").write(skill)

# Comprobación sencilla del frontmatter YAML.
lineas = skill.splitlines()
assert lineas[0] == "---", "Falta el inicio del frontmatter."
assert "name:" in lineas[1], "Falta name en el frontmatter."
assert "name: skill-analista-ventas" in lineas[1], "El nombre de la skill no es el esperado."
assert "description:" in lineas[2], "Falta description en el frontmatter."
assert lineas[3] == "---", "Falta el cierre del frontmatter."
print("Skill creada con frontmatter estándar. Nombre: skill-analista-ventas")
display(Markdown(skill))




## 5. Preparar el prompt y descargar archivos




## Explicación del bloque 5: preparar los archivos

Aquí se escribe un segundo archivo llamado **prompt_para_chatgpt_free.txt**. Este texto explica a ChatGPT qué archivos debe leer y qué análisis debe realizar.

También se muestran los tres archivos que quedan listos:

1. **SKILL.md**: contiene las instrucciones de la skill.
2. **prompt_para_chatgpt_free.txt**: contiene el mensaje para iniciar el análisis.
3. **ventas_demo.csv**: contiene los datos de ejemplo.

Las líneas de descarga están comentadas para que el estudiante pueda ejecutarlas solo cuando las necesite. En Colab se pueden activar quitando el símbolo **#** al inicio.




In [ ]:
prompt = '''Lee el archivo SKILL.md y adopta sus instrucciones para esta conversación.

Analiza ventas_demo.csv aplicando validación de calidad, KPI, comparaciones por periodo, producto y región, hallazgos separados entre hechos e inferencias, recomendaciones priorizadas y nivel de confianza. No inventes datos y explica los supuestos.'''
open("prompt_para_chatgpt_free.txt","w",encoding="utf-8").write(prompt)
print(prompt)
print("\nArchivos listos: SKILL.md, prompt_para_chatgpt_free.txt, ventas_demo.csv")
# En Colab, descomente las descargas:
# from google.colab import files
# files.download("SKILL.md")
# files.download("prompt_para_chatgpt_free.txt")
# files.download("ventas_demo.csv")




## 6. Instalar la skill en ChatGPT

El notebook genera un archivo llamado exactamente **SKILL.md**, que es el nombre que pide el instalador de Skills. La skill tendrá el nombre visible **skill-analista-ventas**.

1. Si ya instaló una versión anterior, elimine la skill **analista-estrategico-ventas** o **skill_analista_ventas**.
2. Ejecute todas las celdas del notebook.
3. Descargue **SKILL.md** sin cambiarle el nombre.
4. Abra ChatGPT y vaya a **Skills**.
5. Pulse el botón **+**.
6. Arrastre **SKILL.md** al cuadro del instalador.
7. Revise que el nombre sea **skill-analista-ventas** y confirme la instalación.

El frontmatter del archivo contiene:

- **name**: nombre corto de la skill.
- **description**: explicación de su propósito.

Si el instalador solicita una carpeta o un archivo .zip, también puede comprimirse el archivo SKILL.md, pero la opción directa recomendada es subir SKILL.md.

